# 09 — Assembling the hearing corpus

**Purpose.** `03` turned 36,378 MODS records into a metadata table. This notebook
adds the transcripts to it and writes the one table the summarization stage reads
from: **one row per hearing, carrying the metadata, the speaker-attributed text,
and the counts needed to decide what is usable**.

**Nothing is filtered here.** Cover-page stubs, hearings with no member roster,
and hearings with no transcript at all stay in the table with columns saying so.
Which of them to drop is a sample-definition decision, and it belongs to the step
that draws the sample, not to the step that builds the corpus. Two facts already
measured make that worth insisting on:

* **A hearing can download perfectly and contain no transcript.** GovInfo serves
  a cover page for hearings it never digitized past the front matter — 1,253
  packages, 3.4 percent, and **99 percent of 1995-96**. Some announce themselves
  with a literal `<TEXT NOT AVAILABLE>`; the rest just stop. `has_marker` and
  `has_text` record the two signals separately.
* **A hearing with no `congMember` cannot pass topic extraction**, which
  attributes statements by matching surnames. That is 4,448 packages, 12.2
  percent. `n_members` makes it visible here rather than at stage 3.

**Two paragraph rules, and the table says which one ran.** GPO marks a paragraph
with a four-space indent, and that rule holds for 99.8 percent of the corpus. A
handful are printed with no indentation at all and separate paragraphs with
blank lines instead. The indent rule cannot see those: it returns the hearing as a
few enormous paragraphs, and because a speaking turn is recognised at the *start*
of a paragraph, the speakers disappear with them. `CHRG-109hhrg30382` holds 185
ordinary speaker lines — `The Chairman.  Good morning.` — none of which falls at a
paragraph boundary under that rule.

`transcript.py` therefore falls back to blank-line splitting when fewer than 5
percent of non-blank lines carry the indent. The two populations are far apart —
hearings that use the convention sit at 17 percent, those that do not at 0.1 to
1.0 — so the threshold is not delicate. On `CHRG-109hhrg30382` the fallback gives
614 paragraphs and 104 speakers against 4 and 0; on `CHRG-109hhrg30793`, 334 and
46 against 48 and 0. A document with no dialogue still yields no speakers, but its
paragraphs are recovered. Across 150 randomly drawn hearings it never fired on the
149 carrying the indent. `paragraph_rule` and `indent_share` record which rule ran
on every row.

**The unit of text is the speaking turn.** `speakers.py` groups paragraphs into
turns, and a turn is what the summarizer must not cut in half — so the turn, not
the paragraph, is what this table stores. Paragraphs survive inside each turn, so
nothing is lost by storing it this way.

**Inputs.**

| Source | Path |
|---|---|
| Hearing metadata | `INTERIM / govinfo / hearings.parquet` — from `03` |
| Member roster | `INTERIM / govinfo / hearing_members.parquet` — from `03` |
| Transcripts | `GOVINFO_RAW / {year} / html / {packageId}.htm` — from `02` |

**Output.**

| File | Path |
|---|---|
| Hearing corpus | `data/raw/02_hearings/hearings_all.parquet` |

## Setup

`lib/` holds every parsing rule; this notebook orchestrates and counts. The one
judgement declared here is `DROP_FRONT_MATTER`, passed through to
`speakers.attribute`: paragraphs carrying a dot leader are table-of-contents and
index rows, and leaving them in makes the first turn of a hearing a page-number
list. The number dropped is recorded per hearing rather than absorbed.

In [1]:
import sys
import time

sys.path.insert(0, "../lib")

import pandas as pd

from paths import GOVINFO_RAW, HEARINGS_REPO, INTERIM
from speakers import attribute
from transcript import has_usable_text, parse_transcript

GOVINFO_INTERIM = INTERIM / "govinfo"
HEARINGS_REPO.mkdir(parents=True, exist_ok=True)

# Dot-leader paragraphs are contents and index rows, not speech.
DROP_FRONT_MATTER = True

hearings = pd.read_parquet(GOVINFO_INTERIM / "hearings.parquet")
members = pd.read_parquet(GOVINFO_INTERIM / "hearing_members.parquet")

# One roster lookup built once. `attribute` reads only bioGuideId and the name,
# but the whole record is carried into the table.
roster = {pid: g.to_dict("records") for pid, g in members.groupby("packageId")}

print(f"hearings : {len(hearings):,}")
print(f"members  : {len(members):,} rows across {len(roster):,} hearings")
print(f"output   : {HEARINGS_REPO}")

hearings : 36,378
members  : 1,031,089 rows across 31,930 hearings
output   : /project2/njia_1089/acroh/projects/llm_method/data/raw/02_hearings


## Step 1 — One hearing

`attribute` returns turns whose paragraphs are a **subsequence** of the input:
front-matter rows are skipped, so turn paragraphs cannot be matched back to their
section label by position alone. Walking the two lists with a single forward
pointer recovers the alignment exactly.

**Every paragraph of a turn is looked up, not just its first.** Questions for the
record are appended to the printed hearing without a speaker opening, so they do
not begin a new turn — they land partway through the turn that was already open,
whose first paragraph is spoken testimony. A turn is therefore marked `qfr` when
**any** of its paragraphs is, and `n_qfr_paragraphs` records how many, so a turn
straddling the boundary stays visible instead of being classified either way.
3,855 hearings carry questions for the record, across 13,139 turns.

In [2]:
def turn_sections(paragraphs: list[str], sections: list[str],
                  turns: list[dict]) -> list[tuple[str, int]]:
    """Per turn, its section label and how many of its paragraphs are `qfr`.

    Recovered by walking the original paragraph list with a forward-only cursor,
    matching every paragraph of every turn. Forward-only means a paragraph that
    repeats within a hearing cannot pull the alignment backwards.
    """
    out = []
    cursor = 0
    for turn in turns:
        labels = []
        for wanted in turn["paragraphs"]:
            while cursor < len(paragraphs):
                if paragraphs[cursor] == wanted:
                    labels.append(sections[cursor])
                    cursor += 1
                    break
                cursor += 1
            else:
                labels.append("spoken")
        n_qfr = sum(1 for value in labels if value == "qfr")
        out.append(("qfr" if n_qfr else "spoken", n_qfr))
    return out


def build_row(package_id: str, year: int) -> dict:
    """Transcript-derived fields for one package, or the empty shape if absent."""
    path = GOVINFO_RAW / str(year) / "html" / f"{package_id}.htm"
    status = has_usable_text(path)

    empty = {
        "turns": [],
        "n_paragraphs": 0, "n_turns": 0, "n_chars": 0, "n_words": 0,
        "n_front_matter": 0, "n_qfr_paragraphs": 0, "n_qfr_turns": 0,
        "n_spoken_chars": 0, "n_paragraphs_dropped": 0,
        "n_turns_unattributed": 0,
        "paragraph_rule": None, "indent_share": 0.0,
        "html_exists": bool(status["exists"]),
        "html_bytes": int(status["bytes"]),
        "has_marker": bool(status["has_marker"]),
        "has_text": bool(status["has_text"]),
    }
    if not status["exists"]:
        return empty

    parsed = parse_transcript(path)
    paragraphs = parsed["paragraphs"]
    if not paragraphs:
        return empty

    grouped = attribute(paragraphs, roster.get(package_id, []),
                        drop_front_matter=DROP_FRONT_MATTER)
    labels = turn_sections(paragraphs, parsed["sections"], grouped)

    turns = []
    kept = 0
    for turn, (label, n_qfr) in zip(grouped, labels):
        text = "\n\n".join(turn["paragraphs"])
        kept += len(turn["paragraphs"])
        turns.append({
            "bioGuideId": turn["bioGuideId"],
            "surname": turn["surname"],
            "honorific": turn["honorific"],
            "is_member": bool(turn["is_member"]),
            "section": label,
            "n_paragraphs": len(turn["paragraphs"]),
            "n_qfr_paragraphs": n_qfr,
            "n_words": sum(len(p.split()) for p in turn["paragraphs"]),
            "text": text,
        })

    return {
        **empty,
        "turns": turns,
        "n_paragraphs": parsed["n_paragraphs"],
        "n_turns": len(turns),
        "n_chars": parsed["n_chars"],
        "n_words": sum(t["n_words"] for t in turns),
        "n_front_matter": parsed["n_front_matter"],
        "n_qfr_paragraphs": parsed["n_qfr_paragraphs"],
        "n_qfr_turns": sum(1 for t in turns if t["section"] == "qfr"),
        "n_spoken_chars": parsed["n_spoken_chars"],
        "paragraph_rule": parsed["paragraph_rule"],
        "indent_share": parsed["indent_share"],
        "n_paragraphs_dropped": parsed["n_paragraphs"] - kept,
        "n_turns_unattributed": sum(1 for t in turns if t["bioGuideId"] is None),
    }

## Step 2 — Walk the corpus

About 18 ms per transcript, measured on a 50-file sample, so the full pass is
roughly ten minutes single-process — cheap enough that parallelising it would add
a failure mode and buy nothing.

In [3]:
started = time.time()
rows = []
for n, hearing in enumerate(hearings.itertuples(), 1):
    rows.append({"packageId": hearing.packageId,
                 **build_row(hearing.packageId, hearing.year)})
    if n % 5000 == 0:
        rate = n / (time.time() - started)
        print(f"  {n:,}/{len(hearings):,}  {rate:.0f} hearings/s  "
              f"eta {(len(hearings) - n) / rate / 60:.1f} min")

text = pd.DataFrame(rows)
elapsed = time.time() - started
print(f"\nparsed {len(text):,} packages in {elapsed / 60:.1f} min")
print(f"  with an HTML file      {int(text['html_exists'].sum()):>7,}")
print(f"  with text in it        {int(text['has_text'].sum()):>7,}")
print(f"  carrying the marker    {int(text['has_marker'].sum()):>7,}")
print(f"  total turns            {int(text['n_turns'].sum()):>7,}")
print(f"  total words            {int(text['n_words'].sum()):>7,}")

  5,000/36,378  48 hearings/s  eta 10.8 min


  10,000/36,378  51 hearings/s  eta 8.6 min


  15,000/36,378  54 hearings/s  eta 6.6 min


  20,000/36,378  55 hearings/s  eta 5.0 min


  25,000/36,378  56 hearings/s  eta 3.4 min


  30,000/36,378  58 hearings/s  eta 1.8 min


  35,000/36,378  58 hearings/s  eta 0.4 min



parsed 36,378 packages in 10.3 min
  with an HTML file       35,503
  with text in it         33,363
  carrying the marker        378
  total turns            7,570,429
  total words            1,160,084,863


## Step 3 — Join to the metadata and write

**`row_group_size` is load-bearing, not tuning.** Arrow addresses string data
with 32-bit offsets, so a column holding more than 2 GB within one row group is
chunked internally, and converting a *nested* chunked column is unimplemented in
pyarrow — `pq.read_table` raises `ArrowNotImplementedError` rather than returning
the data. This table carries 7.6 GB of text inside a list-of-struct column, so as
a single row group it would not be readable at all. At roughly 210 KB of text per
hearing, 2,000 rows is about 420 MB per group, comfortably inside the limit.

The metadata columns are carried through as `03` produced them, plus one derived
here: **`committee_ids`, the flat list of every committee on the record.**

`03` also writes a scalar `committee_id` holding the *primary* committee. Selecting
on that field loses hearings where the committee of interest is listed second, and
763 packages list more than one. Measured against the full list: `committee_id`
finds **940** House Science hearings where the list finds **964**, and **1,414**
Senate Commerce against **1,420** — 30 hidden between them, with both joint
hearings reduced to a single committee.

`lib/mods.py` documents the same hazard one layer down: the collector it replaced
took the first `congCommittee` it found, "so a joint hearing silently became a
hearing of whichever committee happened to be listed first." The nested
`committees` column already holds every entry — checked against
`hearing_committees.parquet` on all 36,378 rows — so this step is a flattening,
not a re-parse.

The id, not the name, is what the sample selection keys on, because
**`authorityId` is stable across committee renames**: `hssy00` covers *Committee
on Science*, *Committee on Science and Technology* and *Committee on Science,
Space, and Technology* as one identifier. A keyword match on
"science|space|technolog" would additionally pick up `sp2k00`, the *Special
Committee on Year 2000 Technology Problem*, which is a different committee.

In [4]:
META = ["packageId", "title", "congress", "session", "chamber", "branch",
        "docClass", "dateIssued", "year", "heldDates", "heldDate_first",
        "committees", "committee_id", "committee_name", "n_committees",
        "members", "witnesses", "n_members", "n_members_with_bioguide",
        "n_witnesses", "has_transcript"]

corpus = hearings[META].merge(text, on="packageId", how="left", validate="one_to_one")

# Every committee on the record, not just the primary one. 53 packages have a
# committee with a null authorityId; those entries are dropped from the flat list
# rather than carried as None, and n_committees still counts them.
corpus["committee_ids"] = corpus["committees"].map(
    lambda v: [c["authorityId"] for c in v if c["authorityId"]] if v is not None else []
)
corpus = corpus.sort_values(["year", "packageId"]).reset_index(drop=True)

hidden = int((corpus["committee_ids"].map(len) > 1).sum())
print(f"committee_ids built; {hidden:,} packages list more than one committee")

# The inherited length rule, kept exactly: gen 2's `1_creatiing_chunks.py` set
#   parsed_txt_char_count = sum(len(p) for p in parsed_txt);  keep if > 8650
# with the comment "Manually Checked that the rows below this number does not
# have conversation text in the transcript". `n_chars` here is that same sum,
# verified identical. It is a character count, not a word count.
#
# `has_text` is an independent signal built from the GovInfo marker and the file
# size. On 2,500 sampled hearings the two rules disagreed on 5 packages, so a row
# must pass BOTH: either signal firing is enough to call the record empty.
MIN_CHARS = 8650

corpus["usable"] = corpus["has_text"] & (corpus["n_chars"] > MIN_CHARS)

n_text = int(corpus["has_text"].sum())
n_len = int((corpus["n_chars"] > MIN_CHARS).sum())
print(f"\nusability, on {len(corpus):,} packages:")
print(f"  has_text                     {n_text:>7,}")
print(f"  n_chars > {MIN_CHARS}              {n_len:>7,}")
print(f"  usable (both)                {int(corpus['usable'].sum()):>7,}  "
      f"({corpus['usable'].mean():.1%})")
print(f"  dropped by has_text only     {int((~corpus['has_text'] & (corpus['n_chars'] > MIN_CHARS)).sum()):>7,}")
print(f"  dropped by length only       {int((corpus['has_text'] & (corpus['n_chars'] <= MIN_CHARS)).sum()):>7,}")

target = HEARINGS_REPO / "hearings_all.parquet"
# See above: one row group would put >2 GB of text in a nested column and make
# the file unreadable. 2,000 rows is ~420 MB per group.
corpus.to_parquet(target, compression="zstd", index=False, row_group_size=2000)

size = target.stat().st_size
import pyarrow.parquet as pq
groups = pq.ParquetFile(target).metadata.num_row_groups
print(f"wrote {target}")
print(f"  {len(corpus):,} rows in {groups} row groups, {size / 1e6:,.0f} MB zstd")
print(f"  {corpus['n_words'].sum() / 1e6:,.0f} million words across "
      f"{int(corpus['n_turns'].sum()):,} turns")
print(f"  columns: {len(corpus.columns)}")

committee_ids built; 763 packages list more than one committee

usability, on 36,378 packages:
  has_text                      33,363
  n_chars > 8650               33,349
  usable (both)                 33,324  (91.6%)
  dropped by has_text only          25
  dropped by length only            39


wrote /project2/njia_1089/acroh/projects/llm_method/data/raw/02_hearings/hearings_all.parquet
  36,378 rows in 19 row groups, 2,504 MB zstd
  1,160 million words across 7,570,429 turns
  columns: 40


## Verification

Eight checks against the written file, read back from disk. The first five are
stop conditions; the rest describe the corpus, because the shape of the coverage
is itself evidence that the usability signal means something.

In [5]:
# Reading the whole file, nested column included, is itself check zero.
back = pd.read_parquet(target)

assert len(back) == len(hearings), f"{len(back):,} rows against {len(hearings):,} packages"
assert not back["packageId"].duplicated().any(), "duplicate packageId"
print(f"1) {len(back):,} rows, one per package, packageId unique")

# The file on disk and the parsed text must agree in both directions.
ghost = back.loc[~back["html_exists"] & (back["n_turns"] > 0)]
silent = back.loc[back["html_exists"] & back["has_text"] & (back["n_turns"] == 0)]
assert len(ghost) == 0, f"{len(ghost)} rows have turns but no HTML file"
print(f"2) no rows carry text without a file; "
      f"{len(silent):,} have a file with text but produced no turn")

bad = back.loc[back["turns"].map(len) != back["n_turns"]]
assert len(bad) == 0, f"{len(bad)} rows where n_turns disagrees with the turn list"
print("3) n_turns matches the stored turn list on every row")

check = back.loc[back["n_turns"] > 0].head(2000)
recomputed = check["turns"].map(lambda ts: sum(t["n_words"] for t in ts))
assert (recomputed == check["n_words"]).all(), "n_words does not match the turns"
print(f"4) n_words recomputed from turns matches on {len(check):,} sampled rows")

# Regression guard: the QFR label was originally taken from a turn's FIRST
# paragraph, which found zero QFR turns on a hearing carrying eleven QFR
# paragraphs, because questions for the record arrive without a speaker opening
# and so land mid-turn. Per-turn counts must never exceed the parse's own count.
qfr = back.loc[back["n_qfr_paragraphs"] > 0]
over = qfr.loc[qfr["turns"].map(lambda ts: sum(t["n_qfr_paragraphs"] for t in ts))
               > qfr["n_qfr_paragraphs"]]
assert len(over) == 0, f"{len(over)} rows attribute more QFR paragraphs than were parsed"
print(f"5) {len(qfr):,} hearings carry questions for the record "
      f"({int(back['n_qfr_turns'].sum()):,} turns); none over-attributed")

print("\n6) paragraph rule, and what the fallback recovered:")
rules = back.loc[back["html_exists"], "paragraph_rule"].value_counts(dropna=False)
print("   " + rules.to_string().replace("\n", "\n   "))
fb = back.loc[back["paragraph_rule"] == "blank_line"]
if len(fb):
    print(f"   fallback fired on {len(fb):,} packages; median {fb['n_paragraphs'].median():,.0f} "
          f"paragraphs and {fb['n_turns'].median():,.0f} turns each")
    stuck = back.loc[back["usable"] & (back["n_chars"] / back["n_paragraphs"].clip(lower=1) > 2000)]
    print(f"   usable rows still averaging >2,000 chars per paragraph: {len(stuck):,}")

print("\n7) usable coverage by year — 1995-96 should be near zero:")
cov = back.groupby("year").agg(hearings=("packageId", "size"),
                               usable=("usable", "sum"),
                               median_words=("n_words", "median"))
cov["pct"] = (cov["usable"] / cov["hearings"]).map("{:.0%}".format)
print(cov.to_string())

print("\n8) committee identifiers survived the round trip:")
# Counted off committee_ids, not the scalar committee_id. The two disagree by
# design and the gap is the whole reason the flat list exists.
for cid, label in [("hssy00", "House Science (all three names)"),
                   ("sscm00", "Senate Commerce, Science, and Transportation")]:
    hit = back["committee_ids"].map(lambda v: cid in v)
    scalar = int(back["committee_id"].eq(cid).sum())
    print(f"   {cid}  {int(hit.sum()):>5,} packages, "
          f"{int(back.loc[hit, 'usable'].sum()):,} usable   {label}")
    print(f"            the primary-committee column alone would find {scalar:,}")
both = back["committee_ids"].map(lambda v: {"hssy00", "sscm00"}.issubset(set(v)))
print(f"   both       {int(both.sum()):>5,} packages carry both committees")
print("\n   No filtering was applied. The next step decides what to drop.")

1) 36,378 rows, one per package, packageId unique
2) no rows carry text without a file; 0 have a file with text but produced no turn
3) n_turns matches the stored turn list on every row
4) n_words recomputed from turns matches on 2,000 sampled rows
5) 3,855 hearings carry questions for the record (13,139 turns); none over-attributed

6) paragraph rule, and what the fallback recovered:
   paragraph_rule
   indent        35381
   blank_line      122
   fallback fired on 122 packages; median 164 paragraphs and 4 turns each
   usable rows still averaging >2,000 chars per paragraph: 49

7) usable coverage by year — 1995-96 should be near zero:
      hearings  usable  median_words  pct
year                                     
1995       253       2         106.0   1%
1996       141       2         111.0   1%
1997       456     354       30228.5  78%
1998       400     280       27665.0  70%
1999       850     734       28659.5  86%
2000       837     762       27893.0  91%
2001      1125   

   hssy00    964 packages, 941 usable   House Science (all three names)
            the primary-committee column alone would find 940
   sscm00  1,420 packages, 1,413 usable   Senate Commerce, Science, and Transportation
            the primary-committee column alone would find 1,414
   both           2 packages carry both committees

   No filtering was applied. The next step decides what to drop.
